# Veyra One — M1 Smoke Test (Kaggle)

Pipeline-Validierung auf Dummy-Daten. GPU erforderlich.


In [ ]:
import torch, sys, os, gc
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Keine GPU! Accelerator setzen.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')

In [ ]:
REPO = '/kaggle/working/veyra'
REPO_URL = 'https://github.com/Lianomeister/Veyra.git'
import shutil
if os.path.exists(REPO): shutil.rmtree(REPO)
!git clone --depth 1 $REPO_URL $REPO
%cd $REPO
assert os.path.exists('training/train_tokenizer.py')
print('repo ok')

In [ ]:
!pip install -q -r requirements.txt
import yaml, omegaconf, transformers
print('deps ok')

In [ ]:
import os
os.makedirs('data/smoke', exist_ok=True)
text = ('The quick brown fox jumps over the lazy dog. ' * 5000)
open('data/smoke/smoke.txt','w').write(text)
print('corpus chars:', len(text))

In [ ]:
!python training/train_tokenizer.py --config configs/eval/lm1b-small.yaml --data data/smoke/smoke.txt --out artifacts/smoke/tokenizer.pt --epochs 1 --batch 4 --max-chunks 20 --lr 1e-3 --amp

In [ ]:
import torch
sd = torch.load('artifacts/smoke/tokenizer.pt', map_location='cpu')
print('tokenizer keys:', list(sd.keys()))
print('num params:', sum(v.numel() for v in sd['model'].values()))
assert 'model' in sd

In [ ]:
torch.cuda.empty_cache(); gc.collect()
!python training/train_generator.py --config configs/eval/lm1b-small.yaml --tokenizer artifacts/smoke/tokenizer.pt --data data/smoke/smoke.txt --out artifacts/smoke/generator.pt --epochs 1 --batch 2 --max-chunks 20 --lr 1e-3 --amp

In [ ]:
import torch
sd = torch.load('artifacts/smoke/generator.pt', map_location='cpu')
print('generator keys:', list(sd.keys()))
print('num params:', sum(v.numel() for v in sd['model'].values()))
assert 'model' in sd

In [ ]:
# Sanity: load model + forward on one batch
import sys; sys.path.insert(0,'.')
import torch, yaml
from omegaconf import OmegaConf
from models.vqvae import VQVAE
from models.next_concept import NextConceptTransformer
cfg = yaml.safe_load(open('configs/eval/lm1b-small.yaml'))
vqvae = VQVAE(**cfg['tokenizer']['vqvae'])
vqvae.load_state_dict(torch.load('artifacts/smoke/tokenizer.pt',map_location='cpu')['model'])
vqvae.eval()
import types
def to_ns(o):
    if isinstance(o,dict): return types.SimpleNamespace(**{k:to_ns(v) for k,v in o.items()})
    if isinstance(o,list): return o
    return o
model = NextConceptTransformer(vqvae, to_ns(cfg['model']))
model.load_state_dict(torch.load('artifacts/smoke/generator.pt',map_location='cpu')['model'])
model.eval()
print('sanity load ok; params:', sum(p.numel() for p in model.parameters()))

In [ ]:
print('ALL SMOKE CHECKS PASSED')
!ls -la artifacts/smoke/